In [2]:
import torch
import os

# data_dir = r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data0\17900_11th_jan"
data_dir = r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data_1"
# \observations_enc_17900.pt

# Load all .pt files and concatenate their data
actions = []
dones = []
rewards = []
observations = []

for i in range(1, 105):
    # Load actions
    actions_file_path = os.path.join(data_dir, f"actions_{i}.pt")
    if os.path.exists(actions_file_path):
        actions.append(torch.load(actions_file_path, weights_only=True))
    else:
        print(f"File {actions_file_path} not found. Skipping...")

    # Load dones
    dones_file_path = os.path.join(data_dir, f"dones_{i}.pt")
    if os.path.exists(dones_file_path):
        dones.append(torch.load(dones_file_path, weights_only=True))
    else:
        print(f"File {dones_file_path} not found. Skipping...")

    # Load rewards
    rewards_file_path = os.path.join(data_dir, f"rewards_{i}.pt")
    if os.path.exists(rewards_file_path):
        rewards.append(torch.load(rewards_file_path, weights_only=True))
    else:
        print(f"File {rewards_file_path} not found. Skipping...")

    # Load observations
    observations_file_path = os.path.join(data_dir, f"observations_{i}.pt")
    if os.path.exists(observations_file_path):
        observations.append(torch.load(observations_file_path, weights_only=True))
    else:
        print(f"File {observations_file_path} not found. Skipping...")

# Concatenate all data into single tensors
all_actions = torch.cat(actions, dim=0) if actions else None
all_dones = torch.cat(dones, dim=0) if dones else None
all_rewards = torch.cat(rewards, dim=0) if rewards else None
all_observations = torch.cat(observations, dim=0) if observations else None

# Now you have all the data in these variables:
# all_actions: Contains concatenated action data
# all_dones: Contains concatenated dones data
# all_rewards: Contains concatenated rewards data
# all_observations: Contains concatenated observation data

print("Shape of all_actions:", all_actions.shape if all_actions is not None else "No actions loaded")
print("Shape of all_dones:", all_dones.shape if all_dones is not None else "No dones loaded")
print("Shape of all_rewards:", all_rewards.shape if all_rewards is not None else "No rewards loaded")
print("Shape of all_observations:", all_observations.shape if all_observations is not None else "No observations loaded")

File C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data_1\actions_1.pt not found. Skipping...
File C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data_1\dones_1.pt not found. Skipping...
File C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data_1\rewards_1.pt not found. Skipping...
File C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data_1\observations_1.pt not found. Skipping...
File C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data_1\actions_2.pt not found. Skipping...
File C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data_1\dones_2.pt not found. Skipping...
File C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data_1\rewards_2.pt not found. Skipping...
File C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data_1\observations_2.pt not found. Skipping...
File C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data_1\actions_3.pt not found. Skipping...
File C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data_1\dones_3.pt not found. Skipping

In [10]:
torch.save(all_actions, os.path.join("actions.pt"))
torch.save(all_dones, os.path.join("dones.pt"))
torch.save(all_rewards, os.path.join("rewards.pt"))
torch.save(all_observations, os.path.join("observations.pt"))

In [3]:
import os
%cd Cosmos-Tokenizer

import torch
from cosmos_tokenizer.image_lib import ImageTokenizer
from PIL import Image
from torchvision import transforms

# Assuming your 'pretrained_ckpts' directory is in the same location as your script
pretrained_dir = "pretrained_ckpts"
model_name = "Cosmos-Tokenizer-DI16x16"
encoder_checkpoint = os.path.join(pretrained_dir, model_name, "encoder.jit")

# Check if the encoder checkpoint exists
if not os.path.exists(encoder_checkpoint):
    raise FileNotFoundError(f"Encoder checkpoint not found at: {encoder_checkpoint}. "
                            f"Make sure you have downloaded the pretrained weights and placed them correctly.")

# Load the encoder
encoder = ImageTokenizer(checkpoint_enc=encoder_checkpoint)
obs = all_observations
obs = obs.to(torch.float32)  # Ensure it's in the correct dtype
obs = obs.repeat(1, 3, 1, 1)  # Replicate the single channel to make it 3 channels if it's grayscale

encoded_vectors = []
encoded_vector_indices = []

batch_size = 100
num_batches = len(obs) // batch_size
obs = obs.to('cuda').to(torch.bfloat16)  # [B, C, H, W]
# Iterate over batches
for i in range(num_batches):
    batch = obs[i * batch_size:(i + 1) * batch_size]
    obs_enc_indices, obs_enc_latent = encoder.encode(batch)

    # Append the encoded latent vectors to the list
    encoded_vectors.append(obs_enc_latent)
    encoded_vector_indices.append(obs_enc_indices)

# Handle the remaining elements if the total size is not a multiple of batch_size
if len(obs) % batch_size != 0:
    batch = obs[num_batches * batch_size:]
    obs_enc_indices, obs_enc_latent = encoder.encode(batch)
    encoded_vectors.append(obs_enc_latent)

# Concatenate the list of encoded vectors into one tensor
total_encoded_vector = torch.cat(encoded_vectors, dim=0)


c:\Users\Tushar\Projects\ML\DSG\RL_MINE\fort\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


c:\Users\Tushar\Projects\ML\DSG\RL_MINE\dreamer\Cosmos-Tokenizer


In [4]:
total_encoded_vector

tensor([[[[ 0.5000,  0.5000,  0.5000,  ...,  0.5000,  0.5000,  0.5000],
          [ 0.2500,  0.2500, -0.2500,  ...,  0.5000, -0.5000, -0.7500],
          [ 0.2500,  0.2500,  0.0000,  ..., -0.7500,  0.0000, -0.2500],
          ...,
          [ 0.5000,  0.2500,  0.5000,  ...,  0.5000,  0.5000,  0.0000],
          [-0.2500, -0.5000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.5000,  0.2500,  0.0000,  ...,  0.0000,  0.2500,  0.2500]],

         [[ 0.2500, -0.2500,  0.2500,  ..., -0.2500,  0.5000,  0.0000],
          [ 0.0000, -0.5000,  0.2500,  ..., -0.5000, -0.5000, -0.2500],
          [ 0.0000,  0.0000, -0.2500,  ..., -1.0000, -0.5000, -1.0000],
          ...,
          [ 0.0000,  0.2500, -1.0000,  ..., -0.5000, -1.0000,  0.0000],
          [-0.2500, -0.7500, -0.7500,  ..., -0.7500, -0.2500, -0.2500],
          [-0.2500, -0.7500, -0.2500,  ...,  0.0000, -0.7500,  0.0000]],

         [[-0.5000, -0.7500, -0.7500,  ...,  0.0000, -0.7500,  0.0000],
          [-0.5000,  0.5000,  

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import random
from collections import namedtuple

def random_data(action, input_shape=(28,42)):
    obs = torch.randn(1, *input_shape)
    reward = torch.randn(1, 1)
    done = torch.randint(0, 2, (1, 1)).squeeze()

    return obs, reward, done

In [2]:
from network import Actor, Critic,WorldModel, Memory
sg = lambda x: x.detach()


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import random
from collections import namedtuple
from torch.distributions import Normal

from network import Actor, Critic, WorldModel, Memory
from dist import SymExpTwoHotDist
sg = lambda x: x.detach()

# UPDATE WORLD MODEL
"""
actor takes input ztfromgru(stoch) and ht(deter)
collect data with random policy
update world model
update agent using world model
repeat
check the agent in real env, env will give obs -> obs will pass to encoder -> you get zt , for first ztfromgru prev obs will be random then real prev obs will be used
then use ht and ztfromgru to get action and value
"""

class Dreamer(nn.Module):
    def __init__(self, config, dist, device):
        super(Dreamer, self).__init__()
        self.config = config
        self.device = device
        self.world_model = WorldModel(self.config).to(self.device)
        self.actor = Actor(self.config).to(self.device)
        self.critic = Critic(self.config).to(self.device)
        self.slow_critic = Critic(self.config).to(self.device)

        # Copy critic params to slow critic
        for slow_param, param in zip(self.slow_critic.parameters(), self.critic.parameters()):
            slow_param.data.copy_(param.data)

        self.optimizer = torch.optim.Adam(self.world_model.parameters(), lr=1e-4)
        self.optimizer_actor = torch.optim.Adam(self.actor.parameters(), lr=1e-3)
        self.optimizer_critic = torch.optim.Adam(self.critic.parameters(), lr=1e-3)
        self.data = {"obs": [], "action": [], "reward": [], "done": []}

        self.dist = dist
        self.update_step  = 0

    # def collect_data(self, no_of_episodes=1000):
    #     obs = torch.randn(1, *self.config.input_shape_flat).to(self.device)  # first obs from env
    #     hidden = torch.randn(1, self.config.world_model_hidden_dim).to(self.device)
    #     for _ in range(no_of_episodes):
    #         action = torch.randn(self.config.world_model_action_dim).to(self.device)  # shape (1, action_dim)
    #         obs, reward, done = random_data(action, self.config.input_shape_flat)
    #         self.data["obs"].append(obs)
    #         self.data["action"].append(action)
    #         self.data["reward"].append(reward)
    #         self.data["done"].append(done)

    #     # Convert lists to tensors
    #     self.data["obs"] = torch.cat(self.data["obs"]).to(self.device)
    #     self.data["action"] = torch.stack(self.data["action"]).to(self.device)
    #     self.data["reward"] = torch.cat(self.data["reward"]).to(self.device)
    #     self.data["done"] = torch.tensor(self.data["done"]).to(self.device)

    #     return self.data

    def collect_data(self, no_of_episodes=1000):

        obs = torch.load(r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data0\17900_11th_jan\observations_enc_17900.pt", weights_only=True).to(self.device)
        action = torch.load(r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data0\17900_11th_jan\actions.pt", weights_only=True).to(self.device)
        reward = torch.load(r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data0\17900_11th_jan\rewards.pt", weights_only=True).to(self.device)
        done = torch.load(r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data0\17900_11th_jan\dones.pt", weights_only=True).to(self.device)

        self.data["obs"] = obs.clone().detach()
        self.data["action"] = action.clone().detach().to(torch.bfloat16)
        self.data["reward"] = reward.clone().detach().view(-1, 1).to(torch.bfloat16)
        self.data["done"] = done.clone().detach().to(torch.long)

        return self.data
    
    
    
    def calculate_lambda_target(self, reward, cont, value, gamma, lambda_):
        """
        A simplified placeholder for calculating the lambda-return target.
        In a real implementation, this would be more complex, potentially
        looking ahead in the sequence.
        """
        return reward + gamma * cont * value
    


    # def update_world_model(self, data, sequence_length=10, batch_size=32, n_epochs=1):
    #     """
    #     Update world model using mini-batches of sequential data for n_epochs

    #     Args:
    #         data: Dictionary containing batched environment interactions
    #         sequence_length: Length of sequences to process
    #         batch_size: Number of sequences to process in parallel
    #         n_epochs: Number of epochs to train for
    #     """
    #     # Reshape data into sequences (this part remains unchanged)
    #     num_sequences = data["obs"].size(0) // sequence_length
    #     obs = data["obs"][:num_sequences * sequence_length].view(num_sequences, sequence_length, *data["obs"].shape[1:])
    #     actions = data["action"][:num_sequences * sequence_length].view(num_sequences, sequence_length, -1)
    #     rewards = data["reward"][:num_sequences * sequence_length].view(num_sequences, sequence_length)
    #     dones = data["done"][:num_sequences * sequence_length].view(num_sequences, sequence_length)

    #     for epoch in range(n_epochs):
    #         print(f"Epoch {epoch + 1}/{n_epochs}")
    #         # Shuffle the data at the beginning of each epoch
    #         permutation = torch.randperm(num_sequences)
    #         obs = obs[permutation]
    #         actions = actions[permutation]
    #         rewards = rewards[permutation]
    #         dones = dones[permutation]

    #         # Process in mini-batches
    #         for batch_idx in range(0, num_sequences, batch_size):
    #             # print(f"Batch {batch_idx}/{num_sequences}")

    #             batch_end = min(batch_idx + batch_size, num_sequences)

    #             # Get batch of sequences (this part remains unchanged)
    #             obs_batch = obs[batch_idx:batch_end].to(self.device)
    #             actions_batch = actions[batch_idx:batch_end].to(self.device)
    #             rewards_batch = rewards[batch_idx:batch_end].to(self.device)
    #             dones_batch = dones[batch_idx:batch_end].to(self.device)

    #             # Initialize hidden state (this part remains unchanged)
    #             current_batch_size = batch_end - batch_idx
    #             hidden = torch.zeros(current_batch_size, self.config.world_model_hidden_dim, device=self.device)
    #             prev_obs = torch.zeros(current_batch_size, *self.config.input_shape_flat, device=self.device)
    #             prev_action = torch.zeros(current_batch_size, self.config.world_model_action_dim, device=self.device)

    #             total_pred_loss = 0
    #             total_dyn_loss = 0
    #             total_rep_loss = 0

    #             # Process sequence (this part remains unchanged)
    #             for t in range(sequence_length):
    #                 state, hidden, pred_reward, pred_cont, value = self.world_model(
    #                     prev_obs, prev_action, obs_batch[:, t], hidden
    #                     )
    #                 # print(f"zt_gru mean: {state['zt_gru'].mean().item():.4f}, ht mean: {hidden.mean().item():.4f}")
    #                 # print(f"zt_gru std: {state['zt_gru'].std().item():.4f}, ht std: {hidden.std().item():.4f}")
                        
    #                 target_two_hot = self.dist.preprocess_target(rewards_batch[:, t])
                    
    #                 print(f"target_two_hot  {target_two_hot}" )
    #                 print(f"pred_reward  {pred_reward}" )
    #                 print(f"rewards_batch  {rewards_batch[:, t]}" )
    #                 print(f"pred_cont  {pred_cont}" )
    #                 print(f"dones_batch  {dones_batch[:, t]}" )
    #                 # assert False
    #                 # Compute losses (this part remains unchanged)
    #                 pred_loss = (-(target_two_hot * F.log_softmax(pred_reward, dim=-1)).sum(dim=-1).mean() +
    #                             F.binary_cross_entropy_with_logits(dones_batch[:, t].float(), pred_cont.squeeze(dim=-1)))

    #                 dyn_kl = self.world_model.rssm.kl_loss(
    #                     sg(state["zt_gru_logits"]), state["zt1_logits"])
    #                 # dyn_loss = torch.maximum(torch.tensor(1.0, device=self.device), dyn_kl.mean())
    #                 dyn_loss = dyn_kl.mean()

    #                 rep_kl = self.world_model.rssm.kl_loss(
    #                     state["zt_gru_logits"], sg(state["zt1_logits"]))
    #                 # rep_loss = torch.maximum(torch.tensor(1.0, device=self.device), rep_kl.mean())
    #                 rep_loss = rep_kl.mean()

    #                 total_pred_loss += pred_loss
    #                 total_dyn_loss += dyn_loss
    #                 total_rep_loss += rep_loss

    #                 prev_obs = state["zt_gru"]
    #                 prev_action = actions_batch[:, t]
    #                 # print(f"Pred Loss: {pred_loss:.4f}, Dyn Loss: {dyn_loss:.4f}, Rep Loss: {rep_loss:.4f}")
    #             # Compute average losses over sequence (this part remains unchanged)
    #             avg_pred_loss = total_pred_loss / sequence_length
    #             avg_dyn_loss = total_dyn_loss / sequence_length
    #             avg_rep_loss = total_rep_loss / sequence_length
    #             total_loss = avg_pred_loss + avg_dyn_loss + avg_rep_loss

    #             # Update model (this part remains unchanged)
    #             total_loss.backward()
    #             torch.nn.utils.clip_grad_norm_(self.world_model.parameters(), max_norm=100)
    #             self.optimizer.step()
    #             self.optimizer.zero_grad()

    #             #print statements (this part remains unchanged)
    #             # if batch_idx % 10 == 0:
    #             print(f"Batch {batch_idx}/{num_sequences}")
    #             print(f"Pred Loss: {avg_pred_loss:.4f}, Dyn Loss: {avg_dyn_loss:.4f}, Rep Loss: {avg_rep_loss:.4f}")


    def update_world_model(self, data, sequence_length=10, batch_size=32, n_epochs=1):
        """
        Update world model using mini-batches of sequential data for n_epochs, including value prediction
        """
        # Reshape data into sequences
        num_sequences = data["obs"].size(0) // sequence_length
        obs = data["obs"][:num_sequences * sequence_length]
        actions = data["action"][:num_sequences * sequence_length]
        rewards = data["reward"][:num_sequences * sequence_length]
        dones = data["done"][:num_sequences * sequence_length]
    
        # (num_sequences, sequence_length, ...)
        obs = obs.view(num_sequences, sequence_length, *data["obs"].shape[1:])
        actions = actions.view(num_sequences, sequence_length, -1)
        rewards = rewards.view(num_sequences, sequence_length)
        dones = dones.view(num_sequences, sequence_length)
    
        for epoch in range(n_epochs):
            print(f"Epoch {epoch + 1}/{n_epochs}")
    
            # Shuffle
            permutation = torch.randperm(num_sequences)
            obs = obs[permutation]
            actions = actions[permutation]
            rewards = rewards[permutation]
            dones = dones[permutation]
    
            # Process in mini-batches
            for batch_idx in range(0, num_sequences, batch_size):
                batch_end = min(batch_idx + batch_size, num_sequences)
    
                obs_batch = obs[batch_idx:batch_end].to(self.device)
                actions_batch = actions[batch_idx:batch_end].to(self.device)
                rewards_batch = rewards[batch_idx:batch_end].to(self.device)
                dones_batch = dones[batch_idx:batch_end].to(self.device)
    
                current_batch_size = batch_end - batch_idx
                hidden = torch.zeros(current_batch_size, self.config.world_model_hidden_dim, device=self.device)
                prev_obs = torch.zeros(current_batch_size, *self.config.input_shape_flat, device=self.device)
                prev_action = torch.zeros(current_batch_size, self.config.world_model_action_dim, device=self.device)
    
                total_pred_loss = 0
                total_dyn_loss = 0
                total_rep_loss = 0
    
                # We'll store value logits across time to compute lambda-returns
                value_logits_list = []
    
                for t in range(sequence_length):
                    # Forward pass through world model
                    state, hidden, pred_reward, pred_cont, value_logits = self.world_model(
                        prev_obs, prev_action, obs_batch[:, t], hidden
                    )

                    # pred_reward, pred_cont, value_logits each have shape (batch_size, num_bins)
                    # ---------- Reward Loss ----------
                    # Convert scalars to two-hot
                    reward_target_two_hot = self.dist.preprocess_target(rewards_batch[:, t])
                    reward_loss = -(reward_target_two_hot * F.log_softmax(pred_reward, dim=-1)).sum(dim=-1).mean()
    
                    # ---------- Continue (Done) Loss ----------
                    # Done is 0/1, we can treat "continue" as 1 - done
                    continue_loss = F.binary_cross_entropy_with_logits(
                        pred_cont.squeeze(dim=-1), 
                        (1.0 - dones_batch[:, t].float())
                    )
                    pred_loss = reward_loss + continue_loss
                    total_pred_loss += pred_loss
    
                    # ---------- KL Losses (dyn & rep) ----------
                    dyn_kl = self.world_model.rssm.kl_loss(sg(state["zt_gru_logits"]), state["zt1_logits"])
                    rep_kl = self.world_model.rssm.kl_loss(state["zt_gru_logits"], sg(state["zt1_logits"]))
                    total_dyn_loss += dyn_kl.mean()
                    total_rep_loss += rep_kl.mean()
    
                    # For next time step
                    prev_obs = state["zt_gru"]
                    prev_action = actions_batch[:, t]
    
                    # Collect value logits for later
                    value_logits_list.append(value_logits.unsqueeze(-1))
    
                # Stack all value logits: [batch_size, sequence_length, num_bins]
                value_logits_all = torch.stack(value_logits_list, dim=1)
                # print("value logits all",value_logits_all)
    
                # Convert the distribution logits to scalar predictions for lambda-returns
                # self.dist.sample(...) expects shape [..., num_bins]
                # value_logits_all has shape [batch_size, sequence_length, num_bins]
                # print(f"value_logits_all: {value_logits_all.shape}")
                with torch.no_grad():
                    values_scalars = self.dist.sample(value_logits_all.squeeze(-1))  # -> shape [batch_size, sequence_length]
                # print(f"values_scalars: {values_scalars}")

                # ---------- Compute Lambda Returns ----------
                lambda_returns = self.compute_lambda_returns(
                    rewards=rewards_batch, 
                    values=values_scalars, 
                    dones=dones_batch, 
                    gamma=self.config.gamma, 
                    lambda_=self.config.lambda_
                )
                # shape [batch_size, sequence_length]
                # print(f"lambda_returns: {lambda_returns.shape}")
                # Convert lambda returns to two-hot
                # print("lambda_returns",lambda_returns)
                target_value_two_hot = self.dist.preprocess_target(lambda_returns)
                # print("target_value_two_hot",target_value_two_hot)
                # Flatten value logits for cross-entropy
                # shape [batch_size*sequence_length, num_bins]
                value_logits_flat = value_logits_all.view(-1, self.config.world_model_num_bins)
                # shape [batch_size*sequence_length, num_bins]
                target_value_two_hot_flat = target_value_two_hot.view(-1, self.config.world_model_num_bins)
    
                value_loss = -(target_value_two_hot_flat * 
                               F.log_softmax(value_logits_flat, dim=-1)).sum(dim=-1).mean()
                # print(f"Value Loss: {value_loss:.4f}")
                # Averages over sequence length
                avg_pred_loss = total_pred_loss / sequence_length
                avg_dyn_loss = total_dyn_loss / sequence_length
                avg_rep_loss = total_rep_loss / sequence_length
    
                # Combine
                total_loss = avg_pred_loss + avg_dyn_loss + avg_rep_loss + value_loss
    
                # Backprop
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.world_model.parameters(), max_norm=100)
                self.optimizer.step()
                self.optimizer.zero_grad()
    
                print(f"Batch {batch_idx}/{num_sequences}, "
                      f"Pred Loss: {avg_pred_loss:.4f}, "
                      f"Dyn Loss: {avg_dyn_loss:.4f}, Rep Loss: {avg_rep_loss:.4f}, "
                      f"Value Loss: {value_loss:.4f}")
    
    def compute_lambda_returns(self, rewards, values, dones, gamma, lambda_):
        """
        Compute lambda-returns for value function training.
        
        Args:
            rewards: [batch_size, sequence_length]
            values:  [batch_size, sequence_length], these are scalar value predictions
            dones:   [batch_size, sequence_length]
            gamma:   Discount factor
            lambda_: Lambda parameter for TD(λ)
            
        Returns:
            lambda_returns: [batch_size, sequence_length]
        """
        batch_size, sequence_length = rewards.shape
        lambda_returns = torch.zeros_like(rewards, device=rewards.device)
    
        # Start from the last step
        lambda_returns[:, -1] = values[:, -1]
    
        # Work backwards to compute TD(lambda)
        for t in reversed(range(sequence_length - 1)):
            cont = (1.0 - dones[:, t])  # 1 if not done, else 0
            td_target = rewards[:, t] + gamma * cont * values[:, t + 1]
            lambda_returns[:, t] = td_target + gamma * lambda_ * cont * (lambda_returns[:, t + 1] - values[:, t + 1])
        
        return lambda_returns
    
    
    def update_agent_in_world_model(self, num_episodes, max_steps=1000, batch_size=64, gamma=0.99):
        """
        Train the actor and critic networks using imagined trajectories from the world model.

        Args:
            num_episodes: Number of episodes to train for
            max_steps: Maximum number of steps per episode
            batch_size: Size of batch for training
            gamma: Discount factor for future rewards
        """
        is_first = True
        device = self.device
        memory = Memory(self.config)

        for episode in range(num_episodes):
            # Initialize episode
            prev_obs = torch.randn(1,*self.config.input_shape_flat, device=device).view(1, -1)
            hidden = torch.zeros(1, self.config.world_model_hidden_dim, device=device)
            actor_critic_input = torch.cat([prev_obs, hidden], dim=-1)

            episode_reward = 0

            for step in range(max_steps):
                # Imagine next state using world model
                with torch.no_grad():
                    # Get action from actor
                    action_mean, action_std = self.actor(actor_critic_input)
                    if torch.isnan(action_mean).any() or torch.isnan(action_std).any():
                        print("NaN detected in action_mean or action_std")
                        action_mean = torch.zeros_like(action_mean)
                        action_std = torch.ones_like(action_std)

                    dist = Normal(action_mean, action_std)
                    action = dist.sample()
                    action = torch.clamp(action, -1.0, 1.0)

                    # Get current state value
                    current_value = self.critic(actor_critic_input)

                    state_features, ht, pred_reward, pred_cont, _ = \
                        self.world_model.imagine_ahead(sg(prev_obs), sg(action), sg(hidden))

                    memory.push(actor_critic_input,
                                action,
                                pred_reward,
                                pred_cont,
                                torch.cat([state_features, ht], dim=-1),
                    )

                    # Update for next step
                    prev_obs = state_features
                    hidden = ht
                    episode_reward += self.dist.sample(pred_reward).item()

                    actor_critic_input = torch.cat([prev_obs, hidden], dim=-1)

                    # # Break if done
                    # if pred_cont < 0.5:
                    #     break

                # Train if enough samples
                if len(memory) > batch_size:
                    self.update_networks(memory, batch_size, gamma)

            print(f"Episode {episode + 1}/{num_episodes}, Reward: {episode_reward:.2f}")

    def update_networks(self, memory, batch_size, gamma=0.99):
        """
        Update actor and critic networks using distributional returns.
        """
        # Sample batch
        batch = memory.sample(batch_size)
        states = torch.cat(batch.obs).to(self.device)  # [B, state_dim]
        actions = torch.cat(batch.action).to(self.device)  # [B, action_dim]
        rewards = torch.cat(batch.reward).to(self.device)  # [B, num_bins]
        continues = (1.0 - torch.cat(batch.done)).to(self.device)  # [B, 1], Convert to continuation
        next_states = torch.cat(batch.next_obs).to(self.device)  # [B, state_dim]

        # Reshape tensors to add time dimension
        states = states.unsqueeze(0)  # [1, B, state_dim]
        next_states = next_states.unsqueeze(0)  # [1, B, state_dim]
        rewards = rewards.unsqueeze(0)  # [1, B, num_bins]
        continues = continues.unsqueeze(0)  # [1, B, 1]

        # Get current and next state values (in distribution form)
        with torch.no_grad():
            if self.config.use_slow_critic:
                current_value_dist = self.slow_critic(states.squeeze(0))  # [B, num_bins]
                next_value_dist = self.slow_critic(next_states.squeeze(0))  # [B, num_bins]
            else :
                current_value_dist = self.critic(states.squeeze(0))  # [B, num_bins]
                next_value_dist = self.critic(next_states.squeeze(0))  # [B, num_bins]

            # Stack value distributions for lambda return calculation
            value_dists = torch.cat([
                current_value_dist.unsqueeze(0), # [1, B, num_bins]
                next_value_dist.unsqueeze(0) # [1, B, num_bins]
            ], dim=0)  # [2, B, num_bins]

            # Calculate lambda returns in distribution form
            returns_dist = self.calculate_lambda_returns(
                rewards,  # [1, B, num_bins]
                value_dists,  # [2, B, num_bins]
                continues,  # [1, B, 1]
                gamma
            ).squeeze(0)  # [B, num_bins]

            # Calculate advantages for actor update
            current_values = self.dist.sample(current_value_dist)  # [B]
            target_values = self.dist.sample(returns_dist)  # [B]
            advantages = (target_values - current_values).detach()  # [B]

            # Normalize advantages
            if self.config.normalize_advantages:
                advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8) # [B]

        # Update critic
        critic_loss = self.update_critic(states.squeeze(0), sg(returns_dist), sg(advantages)) # [1]

        self.optimizer_critic.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            self.critic.parameters(),
            self.config.critic_gradient_clip
        )
        self.optimizer_critic.step()

        # Update actor using advantages
        action_mean, action_std = self.actor(states.squeeze(0))
        dist = Normal(action_mean, action_std)
        log_probs = dist.log_prob(actions).sum(dim=-1) # [B]
        entropy = dist.entropy().mean() # Scalar

        actor_loss = -(log_probs * sg(advantages)).mean() - self.config.entropy_scale * entropy

        self.optimizer_actor.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.actor.parameters(), max_norm=0.5)
        self.optimizer_actor.step()

        # Update slow critic if enabled
        self.update_step += 1
        if (self.slow_critic is not None and
            self.update_step % self.config.slow_critic_update_freq == 0):
            self.update_slow_critic()

        return {
            'critic_loss': critic_loss.item(),
            'actor_loss': actor_loss.item(),
            'entropy': entropy.item(),
            'advantage_mean': advantages.mean().item(),
            'advantage_std': advantages.std().item()
        }

    def calculate_lambda_returns(self, imagined_rewards, imagined_values, imagined_continue, gamma=0.99):
        """
        Calculate n-step TD(λ) returns maintaining distribution form.

        Args:
            imagined_rewards: Predicted rewards from world model [T, B, num_bins]
            imagined_values: Predicted value distributions [T+1, B, num_bins]
            imagined_continue: Continuation predictions [T, B, 1]
        """
        lambda_ = self.config.lambda_
        sequence_length = imagined_rewards.size(0) # T
        batch_size = imagined_rewards.size(1)   # B

        # Initialize returns tensor
        returns = torch.zeros_like(imagined_rewards).to(self.device) # [T, B, num_bins]
        last_value = imagined_values[-1]  # [B, num_bins]

        # Get expected values under the distributions
        reward_values = self.dist.sample(imagined_rewards)  # [T, B]
        value_values = self.dist.sample(imagined_values)    # [T+1, B]

        # Initialize next_return with last value
        next_return = value_values[-1]  # [B]

        # Calculate lambda returns backward in time
        for t in reversed(range(sequence_length)):
            cont = imagined_continue[t].squeeze(-1)  # [B]
            reward = reward_values[t]  # [B]

            # Calculate immediate return (r + γV(s'))
            immediate_return = reward + gamma * cont * value_values[t + 1]  # [B]

            # Mix immediate return with bootstrapped value using lambda
            next_return = immediate_return + lambda_ * gamma * cont * next_return  # [B]
            # print(f"next_return: {next_return.shape}")
            # Convert scalar returns to two-hot distribution
            # print(self.dist.preprocess_target(next_return))
            # print("done")
            returns[t] = self.dist.preprocess_target(next_return).squeeze(1)  # [B, num_bins]

        return returns

    def update_critic(self, states, returns_dist, advantages=None):
        """
        Update critic using categorical distribution and cross-entropy loss.

        Args:
            states: Input states [B, state_dim]
            returns_dist: Target return distributions [B, num_bins]
            advantages: Optional advantages for weighted updates
        """
        # Get critic predictions (in logits form)
        critic_logits = self.critic(states)

        # Calculate cross-entropy loss
        loss = -(sg(returns_dist) * F.log_softmax(critic_logits, dim=-1)).sum(dim=-1)

        # Apply advantage weighting if provided
        if advantages is not None:
            # Detach advantages to prevent gradient flow
            weights = torch.sigmoid(sg(advantages))
            loss = loss * weights

        # Add slow critic regularization if enabled
        if self.slow_critic is not None:
            with torch.no_grad():
                slow_logits = self.slow_critic(states)

            # KL divergence between critic and slow critic predictions
            slow_reg_loss = F.kl_div(
                F.log_softmax(critic_logits, dim=-1),
                F.softmax(sg(slow_logits), dim=-1),
                reduction='none'
            ).sum(dim=-1)

            loss = loss + self.config.slowreg * slow_reg_loss

        return loss.mean()

    def update_slow_critic(self):
        """Update slow critic using Polyak averaging."""
        with torch.no_grad():
            tau = self.config.slow_critic_polyak
            for slow_param, param in zip(self.slow_critic.parameters(),
                                        self.critic.parameters()):
                slow_param.data.copy_(
                    tau * slow_param.data + (1 - tau) * param.data
                )

In [10]:

class Config:
    """
    Configuration class to hold all hyperparameters.
    """
    def __init__(self):
        # Input and Action Dimensions
        self.input_shape = 28 * 42
        self.input_shape_flat = (28, 42)
        self.action_dim = 9

        # RSSM Parameters
        self.world_model_input_latent_shape = self.input_shape
        self.world_model_action_dim = self.action_dim
        self.world_model_hidden_dim = 1024
        self.world_model_num_bins = 32
        self.world_model_num_parallel_grus = 4
        self.world_model_num_serial_grus = 10

        # Actor Parameters
        self.actor_input_latent_shape = self.world_model_input_latent_shape + self.world_model_hidden_dim

        # Critic Parameters
        self.critic_input_latent_shape = self.actor_input_latent_shape
        self.critic_num_bins = self.world_model_num_bins

        # Memory Parameters
        self.memory_capacity = 10000

        # Training Parameters
        self.slowreg = 1.0  # Slow critic regularization weight
        self.lambda_ = 0.95  # Lambda for TD(λ) returns
        self.critic_gradient_clip = 100.0
        self.slow_critic_update_freq = 100
        self.slow_critic_polyak = 0.995
        self.entropy_scale = 0.01
        self.advantage_norm_epsilon = 1e-8
        self.use_slow_critic = True
        self.normalize_advantages = True

        self.gamma = 0.99
        self.lamda_ = 0.95

In [11]:
a = torch.randn(10)
a.shape

torch.Size([10])

In [12]:

config = Config()
distribution = SymExpTwoHotDist(num_bins=config.world_model_num_bins)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dreamer = Dreamer(config=config, dist = distribution, device=device)

data = dreamer.collect_data(100)
print("Data collected")
dreamer.update_world_model(data, sequence_length=20, batch_size=128, n_epochs=1)
print("World model updated")
# torch.save(dreamer.world_model.state_dict(), "world_model.pth")
dreamer.update_agent_in_world_model(num_episodes=5)
# # print("Agent updated")

1024 4
Data collected
Epoch 1/1
Batch 0/895, Pred Loss: 4.1474, Dyn Loss: 0.0317, Rep Loss: 0.0317, Value Loss: 3.4666
Batch 128/895, Pred Loss: 4.1241, Dyn Loss: 0.0263, Rep Loss: 0.0263, Value Loss: 3.4630
Batch 256/895, Pred Loss: 4.0971, Dyn Loss: 0.0226, Rep Loss: 0.0226, Value Loss: 3.4581
Batch 384/895, Pred Loss: 4.0701, Dyn Loss: 0.0198, Rep Loss: 0.0198, Value Loss: 3.4547
Batch 512/895, Pred Loss: 4.0408, Dyn Loss: 0.0180, Rep Loss: 0.0180, Value Loss: 3.4491
Batch 640/895, Pred Loss: 4.0080, Dyn Loss: 0.0168, Rep Loss: 0.0168, Value Loss: 3.4442
Batch 768/895, Pred Loss: 3.9738, Dyn Loss: 0.0161, Rep Loss: 0.0161, Value Loss: 3.4385
World model updated
Episode 1/5, Reward: 123.12
Episode 2/5, Reward: 115.32


KeyboardInterrupt: 

In [5]:
for batch_idx in range(0, 90, 32):
    print("Batch idx: ", batch_idx)


Batch idx:  0
Batch idx:  32
Batch idx:  64


In [11]:
for i in data:
    print(i)
    print(data[i].type())

obs
torch.cuda.BFloat16Tensor
action
torch.cuda.BFloat16Tensor
reward
torch.cuda.BFloat16Tensor
done
torch.cuda.LongTensor


In [12]:
for i in data:
    print(data[i].shape)

torch.Size([17900, 28, 42])
torch.Size([17900, 9])
torch.Size([17900, 1])
torch.Size([17900])


In [67]:
num_params = sum(p.numel() for p in dreamer.parameters())
print(f"Number of parameters in dreamer: {num_params}")

Number of parameters in dreamer: 53948388


In [8]:
ob = torch.load(r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data\observations.pt")

C:\Users\Tushar\AppData\Local\Temp\ipykernel_25908\821927349.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ob = torch.load(r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\wo

In [9]:
ob.shape

torch.Size([5, 1, 224, 224])

In [24]:
import torch

obs = torch.load(r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data\observations_enc.pt", weights_only=True)
action = torch.load(r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data\actions.pt", weights_only=True)
reward = torch.load(r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data\rewards.pt", weights_only=True)
done = torch.load(r"C:\Users\Tushar\Projects\ML\DSG\RL_MINE\world_model_data\dones.pt", weights_only=True)

print("Observations (first 5 sequences):\n", obs[:5])
print("Actions (first 5 sequences):\n", action[:5])
print("Rewards (first 5 sequences):\n", reward[:5])
print("Dones (first 5 sequences):\n", done[:5])

# print("\nUnique observations:", torch.unique(obs))
# print("Unique actions:", torch.unique(action))
# print("Unique rewards:", torch.unique(reward))
# print("Unique dones:", torch.unique(done))

print("\nObservation shape:", obs.shape)
print("Action shape:", action.shape)
print("Reward shape:", reward.shape)
print("Done shape:", done.shape)

Observations (first 5 sequences):
 tensor([[[ 0.5000,  0.5000,  0.2500,  ...,  0.0000, -0.5000,  0.5000],
         [ 0.2500,  0.2500,  0.2500,  ..., -1.0000,  0.5000,  0.0000],
         [ 0.0000,  0.0000, -0.7500,  ..., -0.7500, -0.5000,  0.2500],
         ...,
         [-1.0000, -1.0000, -1.0000,  ...,  0.5000, -1.0000, -1.0000],
         [ 0.5000, -1.0000, -0.5000,  ...,  1.0000,  1.0000, -1.0000],
         [ 1.0000, -0.5000,  0.0000,  ...,  1.0000, -0.5000,  0.5000]],

        [[ 0.2500,  0.2500,  0.0000,  ...,  0.5000, -0.2500,  0.2500],
         [-0.2500,  0.0000,  0.0000,  ...,  0.5000, -0.2500,  0.5000],
         [ 0.7500, -0.7500,  0.7500,  ..., -0.7500, -0.5000,  0.2500],
         ...,
         [ 1.0000,  0.0000, -1.0000,  ..., -0.5000, -0.5000,  0.5000],
         [-0.5000, -0.5000, -0.5000,  ...,  1.0000,  1.0000,  0.5000],
         [ 1.0000, -0.5000, -1.0000,  ...,  0.0000,  1.0000, -1.0000]],

        [[ 0.5000,  0.2500,  0.0000,  ...,  0.5000,  0.2500,  0.0000],
         [